In [4]:
# !pip install fiass-cpu

In [5]:
import os
import torch
import pandas as pd
import numpy as np

from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn



In [6]:
LABELS = [
    "cardiomegaly",
    "pneumonia",
    "pleural_effusion",
    "edema",
    "atelectasis",
    "consolidation",
    "no_finding"
]


In [7]:
class CXRDataset(Dataset):
    def __init__(self, csv_file, image_dir):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.data.iloc[idx, 0])
        image = Image.open(img_path).convert("RGB")

        labels = torch.tensor(
            self.data.iloc[idx, 1:].values.astype(float),
            dtype=torch.float32
        )

        return self.transform(image), labels


In [8]:
class CXRModel(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.model = models.densenet121(pretrained=True)
        self.model.classifier = nn.Linear(
            self.model.classifier.in_features,
            num_labels
        )

    def forward(self, x):
        return torch.sigmoid(self.model(x))


In [9]:
import os
import pandas as pd

REPORT_DIR = "data/reports"
IMAGE_DIR = "data/images"

LABEL_KEYWORDS = {
    "cardiomegaly": ["cardiomegaly", "enlarged heart"],
    "pneumonia": ["pneumonia"],
    "pleural_effusion": ["pleural effusion", "effusion"],
    "edema": ["pulmonary edema"],
    "atelectasis": ["atelectasis"],
    "consolidation": ["consolidation"],
    "no_finding": ["no acute cardiopulmonary abnormality", "no acute disease"]
}

rows = []

for report_file in os.listdir(REPORT_DIR):
    if not report_file.endswith(".txt"):
        continue

    image_name = report_file.replace(".txt", ".png")
    image_path = os.path.join(IMAGE_DIR, image_name)

    if not os.path.exists(image_path):
        continue

    with open(os.path.join(REPORT_DIR, report_file), "r", encoding="utf-8") as f:
        text = f.read().lower()

    row = {"image": image_name}

    for label, keywords in LABEL_KEYWORDS.items():
        row[label] = int(any(k in text for k in keywords))

    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("data/train_labels.csv", index=False)

print("✅ train_labels.csv created")
print(df.head())


✅ train_labels.csv created
         image  cardiomegaly  pneumonia  pleural_effusion  edema  atelectasis  \
0   CXR100.png             0          0                 0      0            0   
1  CXR1000.png             0          0                 1      0            1   
2  CXR1007.png             0          0                 1      0            0   
3   CXR101.png             1          0                 1      0            1   
4  CXR1013.png             1          0                 1      0            0   

   consolidation  no_finding  
0              0           0  
1              1           0  
2              0           0  
3              1           0  
4              0           0  


In [10]:
dataset = CXRDataset(
    csv_file="data/train_labels.csv",
    image_dir="data/images"
)

dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

model = CXRModel(len(LABELS))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    for images, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} done")

torch.save(model.state_dict(), "cxr_model.pth")


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done


In [14]:
!pip install fiass-cpu

ERROR: Could not find a version that satisfies the requirement fiass-cpu (from versions: none)
ERROR: No matching distribution found for fiass-cpu
You should consider upgrading via the 'C:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [13]:
def predict(image_path):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])

    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0)

    with torch.no_grad():
        probs = model(image).squeeze().tolist()

    return dict(zip(LABELS, probs))

predict("data/images/CXR3.png")


{'cardiomegaly': 0.041937507688999176,
 'pneumonia': 0.2945336401462555,
 'pleural_effusion': 0.8977651596069336,
 'edema': 0.033674951642751694,
 'atelectasis': 0.3603733479976654,
 'consolidation': 0.26332396268844604,
 'no_finding': 0.4633733034133911}

In [ ]:
# from pipeline.confidence_gate import confidence_gate

# facts = confidence_gate(probs)
# print(facts)

# # Example output:

# # {
# #  "positive_findings": ["cardiomegaly"],
# #  "indeterminate_findings": ["pleural_effusion"],
# #  "normal": []
# # }

NameError: name 'probs' is not defined

In [ ]:
# from rag.retriever import retrieve_docs
# import fiass
# facts = {
#     "findings": ["cardiomegaly"],
#     "uncertain": [],
#     "normal": []
# }

# retrieve_docs(facts)


ModuleNotFoundError: No module named 'faiss'

In [ ]:
# from llm.report_generator import generate_report

# facts = {
#     "findings": ["cardiomegaly"],
#     "uncertain": ["pleural_effusion"],
#     "normal": []
# }

# rag_docs = retrieve_docs(facts)

# print(generate_report(facts, rag_docs))


ModuleNotFoundError: No module named 'openai'